# 03. Causal Inference: 2SLS IV, Difference-in-Differences, and Diagnostics

Recover structural elasticity using `supplier_cost_shock` as an IV and evaluate the regional DiD policy rollout.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from simulation.data_generator import generate_synthetic_marketplace_data
from preprocessing.cleaning import clean_transaction_data
from preprocessing.feature_engineering import add_engineered_features
from causal.iv import estimate_2sls_iv
from causal.did import estimate_difference_in_differences
from causal.diagnostics import run_causal_diagnostics_report

feat_df = add_engineered_features(clean_transaction_data(generate_synthetic_marketplace_data(n_rows=15000, seed=42))[0])
iv_res = estimate_2sls_iv(feat_df)
did_res = estimate_difference_in_differences(feat_df)
print(f'2SLS IV Elasticity: {iv_res.elasticity:.4f} (SE={iv_res.std_error:.4f}) vs True Mean: {feat_df["true_elasticity"].mean():.4f}')
print(f'First-stage F: {iv_res.first_stage_f_stat:.2f} | Wu-Hausman p: {iv_res.wu_hausman_p_value:.2e}')
print(f'DiD ATE (log Q): {did_res.ate_log_quantity:.4f} | Implied Elasticity: {did_res.implied_did_elasticity:.4f} | Pre-trend p: {did_res.pre_trend_p_value:.4f}')

2026-11-27 05:26:29 | INFO     | preprocessing.cleaning | Cleaned dataset: 15000 -> 15000 rows (dups=0, invalid=0, imputed=0, clipped=45)


2SLS IV Elasticity: -1.0118 (SE=0.0133) vs True Mean: -1.0195
First-stage F: 44967.30 | Wu-Hausman p: 0.00e+00
DiD ATE (log Q): 0.2129 | Implied Elasticity: -1.5060 | Pre-trend p: 0.2057


In [2]:
did_res.event_study_df

,relative_period,estimate,std_error,ci_low,ci_high,p_value
0,-6,0.007650,0.045566,-0.081658,0.096958,8.666674e-01
1,-5,0.023174,0.044947,-0.064920,0.111268,6.061420e-01
2,-4,0.017006,0.046269,-0.073680,0.107691,7.132174e-01
3,-3,0.008322,0.046058,-0.081950,0.098594,8.566217e-01
4,-2,-0.050082,0.046010,-0.140260,0.040096,2.763766e-01
5,-1,0.000000,0.000000,0.000000,0.000000,1.000000e+00
6,0,0.206932,0.046010,0.116755,0.297110,6.873229e-06
7,1,0.223192,0.045090,0.134816,0.311567,7.426604e-07
8,2,0.212179,0.045375,0.123246,0.301113,2.923429e-06
9,3,0.271198,0.046786,0.179499,0.362898,6.769555e-09


In [3]:
diag = run_causal_diagnostics_report(feat_df, iv_result=iv_res, did_result=did_res)
diag['assumptions_registry'][['method', 'assumption', 'testable_from_observational_data']]

,method,assumption,testable_from_observational_data
0,IV (2SLS),"Relevance: Cov(Z, log P | X) != 0",True
1,IV (2SLS),Exclusion Restriction: Z affects Q only throug...,False
2,IV (2SLS),Independence: Z is independent of unobserved d...,False
3,Difference-in-Differences,Parallel Trends: Treated & control groups shar...,False
4,Difference-in-Differences,No Anticipation / Sharp Policy Timing,False
5,CATE / Uplift (T- & X-Learner),Conditional Ignorability / Unconfoundedness: (...,False
6,CATE / Uplift / Policy,Positivity (Overlap): 0 < P(T=1 | X=x) < 1,True
7,All Estimators,SUTVA: No interference between units & well-de...,False
